# Information Theory Diagnostics for Cellular Infrastructure

This notebook computes and visualizes information theory metrics on cellular telemetry from Airtel Rwanda. We investigate:
1. **Shannon Entropy** of diurnal traffic volumes to analyze network demand predictability.
2. **Kullback-Leibler (KL) Divergence** to compare traffic profile variations between weekdays and weekends.
3. **Mutual Information** between Radio Access Technology (RAT) generations and traffic types (uplink vs. downlink, payload vs. control signaling).
4. **RRC State space & Feedback States** estimation.
5. **Satellite RTT Wall & Channel Dispersion** calculation.
6. **Peak Age of Information (AoI) Optimization** over core signaling queues.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import fynesse
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.dpi'] = 120

## 1. Data Ingestion
We load the aggregated temporal trace of TCP/UDP flows.

In [ ]:
df = fynesse.load_joined_temporal_data()
df.head()

## 2. Diurnal Demand Predictability (Shannon Entropy)
We compute the Shannon Entropy $H(X)$ of hourly traffic volume distribution over the day:
$$H(X) = - \sum_{h=0}^{23} p_h \log_2 p_h$$
where $p_h$ is the proportion of total traffic volume in hour $h$.

In [ ]:
entropy = fynesse.calculate_hourly_traffic_entropy(df)

## 3. Weekday vs. Weekend Diurnal Divergence (Kullback-Leibler Divergence)
We measure the relative entropy difference between weekday hourly traffic distribution ($P$) and weekend hourly traffic distribution ($Q$):
$$D_{\text{KL}}(P \parallel Q) = \sum_{h=0}^{23} P_h \log_2 \left(\frac{P_h}{Q_h}\right)$$

In [ ]:
kl_div = fynesse.calculate_weekday_weekend_kl_divergence(df)

## 4. RAT vs. Traffic Code (Mutual Information)
We evaluate how much information the choice of Radio Access Technology (RAT) shares with the type of traffic (uplink vs. downlink, payload vs. signaling):
$$I(RAT; Code) = \sum_{x \in RAT} \sum_{y \in Code} p(x,y) \log_2 \left(\frac{p(x,y)}{p(x)p(y)}\right)$$

In [ ]:
mutual_info = fynesse.calculate_rat_traffic_code_mutual_information(df)

## 5. RRC State space & Feedback States
We estimate the stationary distribution $\mathbf{P}_S$ of Dedicated (low delay), Shared (RLC reordering), and Idle (packet drop/retransmission) states.

In [ ]:
qos_df = fynesse.load_qos_metrics()
p_s = fynesse.estimate_feedback_channel_states(qos_df)

## 6. Satellite RTT Wall & Channel Dispersion
We extract the RTT step-function above 200ms and compute the empirical channel dispersion $V_{\text{sat}}$ in seconds squared.

In [ ]:
wan_df = fynesse.load_rtt_data(sheet_name="wan")
mean_rtt, var_rtt = fynesse.estimate_satellite_dispersion(wan_df)

## 7. Peak Age of Information (AoI) Optimization
We process core GTP-C request timelines to calculate arrival rate $\lambda_{\text{sig}}$ and solve the $M/GI/1/K$ queueing-theoretic model to locate the optimal arrival rate $\lambda^*$ that minimizes information staleness.

In [ ]:
gtpc_df = fynesse.load_gtpc_signaling()
opt_lambda, opt_aoi = fynesse.estimate_core_signaling_aoi(gtpc_df)